In [111]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sagorkumarmitra/nlp-shakespeare/shakespeare.txt


In [112]:
path = kagglehub.dataset_download("sagorkumarmitra/nlp-shakespeare")

In [113]:
import torch 
from torch import nn
import torch.nn.functional as F

import numpy as np
import matplotlib.pyplot as plt

In [114]:
with open('/kaggle/input/datasets/sagorkumarmitra/nlp-shakespeare/shakespeare.txt','r',encoding='utf8') as f:
    text = f.read()

In [115]:
type(text)

str

In [116]:
print(text[:1000])


                     1
  From fairest creatures we desire increase,
  That thereby beauty's rose might never die,
  But as the riper should by time decease,
  His tender heir might bear his memory:
  But thou contracted to thine own bright eyes,
  Feed'st thy light's flame with self-substantial fuel,
  Making a famine where abundance lies,
  Thy self thy foe, to thy sweet self too cruel:
  Thou that art now the world's fresh ornament,
  And only herald to the gaudy spring,
  Within thine own bud buriest thy content,
  And tender churl mak'st waste in niggarding:
    Pity the world, or else this glutton be,
    To eat the world's due, by the grave and thee.


                     2
  When forty winters shall besiege thy brow,
  And dig deep trenches in thy beauty's field,
  Thy youth's proud livery so gazed on now,
  Will be a tattered weed of small worth held:  
  Then being asked, where all thy beauty lies,
  Where all the treasure of thy lusty days;
  To say within thine own deep su

In [117]:
len(text)

5445609

In [118]:
all_characters = set(text)

In [119]:
print(all_characters)

{'F', '-', 'R', 'f', 'I', '8', 'W', '0', '"', 'h', '(', 'E', 'P', 'a', ']', ':', 'y', 'z', 'j', 'U', 'V', '5', 'X', 'k', '}', '6', 'M', 'b', 'm', '2', 'w', 'J', '<', 'c', 'S', '\n', ',', ';', 'q', '9', 'O', ' ', '7', '3', 'C', 'T', '?', 't', 'n', 'p', 'N', 'o', '&', 'l', 'v', '1', 'B', 'A', 'Q', 'G', 'L', 'Y', "'", 's', '[', '4', ')', 'g', 'x', 'u', 'Z', '.', '_', '|', 'r', 'i', 'D', '`', 'H', 'd', 'K', 'e', '>', '!'}


In [120]:
len(all_characters)

84

In [121]:
# num - letter
decoder = dict(enumerate(all_characters))

In [122]:
encode = {char: ind for ind, char in decoder.items()}

In [123]:
print(decoder)

{0: 'F', 1: '-', 2: 'R', 3: 'f', 4: 'I', 5: '8', 6: 'W', 7: '0', 8: '"', 9: 'h', 10: '(', 11: 'E', 12: 'P', 13: 'a', 14: ']', 15: ':', 16: 'y', 17: 'z', 18: 'j', 19: 'U', 20: 'V', 21: '5', 22: 'X', 23: 'k', 24: '}', 25: '6', 26: 'M', 27: 'b', 28: 'm', 29: '2', 30: 'w', 31: 'J', 32: '<', 33: 'c', 34: 'S', 35: '\n', 36: ',', 37: ';', 38: 'q', 39: '9', 40: 'O', 41: ' ', 42: '7', 43: '3', 44: 'C', 45: 'T', 46: '?', 47: 't', 48: 'n', 49: 'p', 50: 'N', 51: 'o', 52: '&', 53: 'l', 54: 'v', 55: '1', 56: 'B', 57: 'A', 58: 'Q', 59: 'G', 60: 'L', 61: 'Y', 62: "'", 63: 's', 64: '[', 65: '4', 66: ')', 67: 'g', 68: 'x', 69: 'u', 70: 'Z', 71: '.', 72: '_', 73: '|', 74: 'r', 75: 'i', 76: 'D', 77: '`', 78: 'H', 79: 'd', 80: 'K', 81: 'e', 82: '>', 83: '!'}


In [124]:
print(encode)

{'F': 0, '-': 1, 'R': 2, 'f': 3, 'I': 4, '8': 5, 'W': 6, '0': 7, '"': 8, 'h': 9, '(': 10, 'E': 11, 'P': 12, 'a': 13, ']': 14, ':': 15, 'y': 16, 'z': 17, 'j': 18, 'U': 19, 'V': 20, '5': 21, 'X': 22, 'k': 23, '}': 24, '6': 25, 'M': 26, 'b': 27, 'm': 28, '2': 29, 'w': 30, 'J': 31, '<': 32, 'c': 33, 'S': 34, '\n': 35, ',': 36, ';': 37, 'q': 38, '9': 39, 'O': 40, ' ': 41, '7': 42, '3': 43, 'C': 44, 'T': 45, '?': 46, 't': 47, 'n': 48, 'p': 49, 'N': 50, 'o': 51, '&': 52, 'l': 53, 'v': 54, '1': 55, 'B': 56, 'A': 57, 'Q': 58, 'G': 59, 'L': 60, 'Y': 61, "'": 62, 's': 63, '[': 64, '4': 65, ')': 66, 'g': 67, 'x': 68, 'u': 69, 'Z': 70, '.': 71, '_': 72, '|': 73, 'r': 74, 'i': 75, 'D': 76, '`': 77, 'H': 78, 'd': 79, 'K': 80, 'e': 81, '>': 82, '!': 83}


In [125]:
encoded_text = np.array([encode[char] for char in text])

In [126]:
encoded_text[:900]

array([35, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41,
       41, 41, 41, 41, 41, 55, 35, 41, 41,  0, 74, 51, 28, 41,  3, 13, 75,
       74, 81, 63, 47, 41, 33, 74, 81, 13, 47, 69, 74, 81, 63, 41, 30, 81,
       41, 79, 81, 63, 75, 74, 81, 41, 75, 48, 33, 74, 81, 13, 63, 81, 36,
       35, 41, 41, 45,  9, 13, 47, 41, 47,  9, 81, 74, 81, 27, 16, 41, 27,
       81, 13, 69, 47, 16, 62, 63, 41, 74, 51, 63, 81, 41, 28, 75, 67,  9,
       47, 41, 48, 81, 54, 81, 74, 41, 79, 75, 81, 36, 35, 41, 41, 56, 69,
       47, 41, 13, 63, 41, 47,  9, 81, 41, 74, 75, 49, 81, 74, 41, 63,  9,
       51, 69, 53, 79, 41, 27, 16, 41, 47, 75, 28, 81, 41, 79, 81, 33, 81,
       13, 63, 81, 36, 35, 41, 41, 78, 75, 63, 41, 47, 81, 48, 79, 81, 74,
       41,  9, 81, 75, 74, 41, 28, 75, 67,  9, 47, 41, 27, 81, 13, 74, 41,
        9, 75, 63, 41, 28, 81, 28, 51, 74, 16, 15, 35, 41, 41, 56, 69, 47,
       41, 47,  9, 51, 69, 41, 33, 51, 48, 47, 74, 13, 33, 47, 81, 79, 41,
       47, 51, 41, 47,  9

In [127]:
decoder[78]

'H'

In [128]:
def one_hot_encoder(encoded_text,num_uni_chars):
    # encoded_text > batch of encoded text
    # num_uni_chars > len(set(text))
    one_hot = np.zeros((encoded_text.size,num_uni_chars))#encoded text its a numpy array
    one_hot = one_hot.astype(np.float32)
    one_hot[np.arange(one_hot.shape[0]),encoded_text.flatten()] = 1.0
    one_hot =one_hot.reshape((*encoded_text.shape,num_uni_chars))
    return one_hot

In [129]:
def generate_batches(encoded_text,samp_per_batch=10,seq_len=50):
    # amount of char per batch
    char_per_batch = samp_per_batch * seq_len
    num_batches_avail= int(len(encoded_text)/char_per_batch)
    # amount of batches we can make, given the len of encoded_text
    encoded_text = encoded_text[:num_batches_avail*char_per_batch]
    encoded_text = encoded_text.reshape((samp_per_batch,-1))

    for n in range(0,encoded_text.shape[1],seq_len):
        x = encoded_text[:,n:n+seq_len]
        #zeros array to the same shape as x
        y = np.zeros_like(x)

        try:
            y[:,:-1] = x[:,1:]
            y[:,-1] = encoded_text[:,n+seq_len]
        except:
            y[:,:-1] = x [:,1:]
            y[:,-1] = encoded_text[:,0]
        yield x,y
        

In [130]:
sample_text = encoded_text[:20]
sample_text

array([35, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41,
       41, 41, 41])

In [131]:
sample_text = encoded_text[:20]
sample_text

array([35, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41, 41,
       41, 41, 41])

In [132]:
batch_generator = generate_batches(sample_text,samp_per_batch=2,seq_len=5)

In [133]:
x,y = next(batch_generator)

In [134]:
x,y

(array([[35, 41, 41, 41, 41],
        [41, 41, 41, 41, 41]]),
 array([[41, 41, 41, 41, 41],
        [41, 41, 41, 41, 41]]))

In [135]:
class CharModel(nn.Module):
    
    def __init__(self, all_chars, num_hidden=256, num_layers=4,drop_prob=0.5,use_gpu=False):
        
        
        # SET UP ATTRIBUTES
        super().__init__()
        self.drop_prob = drop_prob
        self.num_layers = num_layers
        self.num_hidden = num_hidden
        self.use_gpu = use_gpu
        
        #CHARACTER SET, ENCODER, and DECODER
        self.all_chars = all_chars
        self.decoder = dict(enumerate(all_chars))
        self.encoder = {char: ind for ind,char in decoder.items()}
        
        
        self.lstm = nn.LSTM(len(self.all_chars), num_hidden, num_layers, dropout=drop_prob, batch_first=True)
        
        self.dropout = nn.Dropout(drop_prob)
        
        self.fc_linear = nn.Linear(num_hidden, len(self.all_chars))
      
    
    def forward(self, x, hidden):
                  
        
        lstm_output, hidden = self.lstm(x, hidden)
        
        
        drop_output = self.dropout(lstm_output)
        
        drop_output = drop_output.contiguous().view(-1, self.num_hidden)
        
        
        final_out = self.fc_linear(drop_output)
        
        
        return final_out, hidden
    
    
    def hidden_state(self, batch_size):
        '''
        Used as separate method to account for both GPU and CPU users.
        '''
        
        if self.use_gpu:
            
            hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden).cuda(),
                     torch.zeros(self.num_layers,batch_size,self.num_hidden).cuda())
        else:
            hidden = (torch.zeros(self.num_layers,batch_size,self.num_hidden),
                     torch.zeros(self.num_layers,batch_size,self.num_hidden))
        
        return hidden
        

In [136]:
  model = CharModel(all_chars = all_characters,
                   num_hidden = 512,
                   num_layers = 3,
                   drop_prob= 0.5,
                   use_gpu = True)

In [137]:
total_param=[]
for p in model.parameters():
    total_param.append(int(p.numel()))
print(sum(total_param))

5470292


In [138]:
optimizer= torch.optim.Adam(model.parameters(),lr=0.001)
criterion = nn.CrossEntropyLoss()

In [139]:
train_percent = 0.1

In [140]:
train_ind = int(len(encoded_text)*train_percent)

In [141]:
train_data = encoded_text[:train_ind]
val_data = encoded_text[train_ind:]

In [142]:
len(train_data)

544560

In [143]:
train_percent = 0.9
train_ind = int(len(encoded_text)* train_percent)
train_data = encoded_text[:train_ind]
val_data = encoded_text[train_ind:]

In [144]:
epochs = 60
batch_size = 100

seq_len=100

tracker =0

num_char = max(encoded_text)+1

In [145]:
# Set model to train
model.train()


# Check to see if using GPU
if model.use_gpu:
    model.cuda()

for i in range(epochs):
    
    hidden = model.hidden_state(batch_size)
    
    
    for x,y in generate_batches(train_data,batch_size,seq_len):
        
        tracker += 1
        
        # One Hot Encode incoming data
        x = one_hot_encoder(x,num_char)
        
        # Convert Numpy Arrays to Tensor
        
        inputs = torch.from_numpy(x)
        targets = torch.from_numpy(y)
        
        # Adjust for GPU if necessary
        
        if model.use_gpu:
            
            inputs = inputs.cuda()
            targets = targets.cuda()
            
        # Reset Hidden State
        # If we dont' reset we would backpropagate through all training history
        hidden = tuple([state.data for state in hidden])
        
        model.zero_grad()
        
        lstm_output, hidden = model.forward(inputs,hidden)
        loss = criterion(lstm_output,targets.view(batch_size*seq_len).long())
        
        loss.backward()
        
        # POSSIBLE EXPLODING GRADIENT PROBLEM!
        # LET"S CLIP JUST IN CASE
        nn.utils.clip_grad_norm_(model.parameters(),max_norm=5)
        
        optimizer.step()
        
        
        
        ###################################
        ### CHECK ON VALIDATION SET ######
        #################################
        
        if tracker % 25 == 0:
            
            val_hidden = model.hidden_state(batch_size)
            val_losses = []
            model.eval()
            
            for x,y in generate_batches(val_data,batch_size,seq_len):
                
                # One Hot Encode incoming data
                x = one_hot_encoder(x,num_char)
                

                # Convert Numpy Arrays to Tensor

                inputs = torch.from_numpy(x)
                targets = torch.from_numpy(y)

                # Adjust for GPU if necessary

                if model.use_gpu:

                    inputs = inputs.cuda()
                    targets = targets.cuda()
                    
                # Reset Hidden State
                # If we dont' reset we would backpropagate through 
                # all training history
                val_hidden = tuple([state.data for state in val_hidden])
                
                lstm_output, val_hidden = model.forward(inputs,val_hidden)
                val_loss = criterion(lstm_output,targets.view(batch_size*seq_len).long())
        
                val_losses.append(val_loss.item())
            
            # Reset to training model after val for loop
            model.train()
            
            print(f"Epoch: {i} Step: {tracker} Val Loss: {val_loss.item()}")

Epoch: 0 Step: 25 Val Loss: 3.204157829284668
Epoch: 0 Step: 50 Val Loss: 3.19325852394104
Epoch: 0 Step: 75 Val Loss: 3.1907498836517334
Epoch: 0 Step: 100 Val Loss: 3.18738055229187
Epoch: 0 Step: 125 Val Loss: 3.0859391689300537
Epoch: 0 Step: 150 Val Loss: 2.983184337615967
Epoch: 0 Step: 175 Val Loss: 2.9209561347961426
Epoch: 0 Step: 200 Val Loss: 2.7574069499969482
Epoch: 0 Step: 225 Val Loss: 2.6810109615325928
Epoch: 0 Step: 250 Val Loss: 2.607102394104004
Epoch: 0 Step: 275 Val Loss: 2.4689781665802
Epoch: 0 Step: 300 Val Loss: 2.3597934246063232
Epoch: 0 Step: 325 Val Loss: 2.2754571437835693
Epoch: 0 Step: 350 Val Loss: 2.213123083114624
Epoch: 0 Step: 375 Val Loss: 2.161747694015503
Epoch: 0 Step: 400 Val Loss: 2.118105888366699
Epoch: 0 Step: 425 Val Loss: 2.080328941345215
Epoch: 0 Step: 450 Val Loss: 2.0499343872070312
Epoch: 0 Step: 475 Val Loss: 2.0119545459747314
Epoch: 1 Step: 500 Val Loss: 1.98854398727417
Epoch: 1 Step: 525 Val Loss: 1.964965581893921
Epoch: 1 Ste

In [146]:
model_name= "model.pt"

In [147]:
torch.save(model.state_dict(),model_name)

In [148]:

model = CharModel(
    all_chars=all_characters,
    num_hidden=512,
    num_layers=3,
    drop_prob=0.5,
    use_gpu=True,
)

In [149]:
model.load_state_dict(torch.load(model_name))

<All keys matched successfully>

In [150]:
model.eval()

CharModel(
  (lstm): LSTM(84, 512, num_layers=3, batch_first=True, dropout=0.5)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc_linear): Linear(in_features=512, out_features=84, bias=True)
)

In [151]:
def predict_next_char(model, char, hidden=None, k=1):
        
        # Encode raw letters with model
        encoded_text = model.encoder[char]
        
        encoded_text = np.array([[encoded_text]])
        
        encoded_text = one_hot_encoder(encoded_text, len(model.all_chars))
        
        inputs = torch.from_numpy(encoded_text)
        
        # Check for CPU
        if(model.use_gpu):
            inputs = inputs.cuda()
        
        
        # Grab hidden states
        hidden = tuple([state.data for state in hidden])
        
        
        # Run model and get predicted output
        lstm_out, hidden = model(inputs, hidden)

        
        # Convert lstm_out to probabilities
        probs = F.softmax(lstm_out, dim=1).data
        
        
        
        if(model.use_gpu):
            # move back to CPU to use with numpy
            probs = probs.cpu()
        
        
        
        # Return k largest probabilities in tensor
        probs, index_positions = probs.topk(k)
        
        
        index_positions = index_positions.numpy().squeeze()
        
        # Create array of probabilities
        probs = probs.numpy().flatten()
        
        # Convert to probabilities per index
        probs = probs/probs.sum()
        
        # randomly choose a character based on probabilities
        char = np.random.choice(index_positions, p=probs)
       
        # return the encoded value of the predicted char and the hidden state
        return model.decoder[char], hidden

In [152]:
def generate_text(model, size, seed='The', k=1):
        
      
    
    
    if(model.use_gpu):
        model.cuda()
    else:
        model.cpu()
    
    # Evaluation mode
    model.eval()
    
    # begin output from initial seed
    output_chars = [c for c in seed]
    
    # intiate hidden state
    hidden = model.hidden_state(1)
    
    # predict the next character for every character in seed
    for char in seed:
        char, hidden = predict_next_char(model, char, hidden, k=k)
    
    # add initial characters to output
    output_chars.append(char)
    
    # Now generate for size requested
    for i in range(size):
        
        # predict based off very last letter in output_chars
        char, hidden = predict_next_char(model, output_chars[-1], hidden, k=k)
        
        # add predicted character
        output_chars.append(char)
    
    # return string of predicted text
    return ''.join(output_chars)

In [153]:
print(generate_text(model, 1000, seed='The ', k=3))

The SIR WILLIAMS BURGUNDY, as the Duke of Saint Capulet
    and the Duke of Westmoreland

  KING RICHARD THE SERVANT to Allagon

  LUCY, sir, a contention
  ANGELO, a stone of the Duke
  SERVANT and COUNTESS

  PANDARUS
                                                       EARL OF SURREY

  KING JOHN, the Duke of Britaine
  SALERIO, a courtier to the King
  SIR HUGH EVANS
  CAIUS
  PRINCESS OF SHALLOW, at a fresh son
  SIR HUGH EVANS  
  CADE, a trumpet

    Sir, a monster to the Duke of Warwick, and a song, and the King
  SALISBURY, his son to the Duke

    So the Duke of Gloucester shakes him
                                   and the KING, and all
                                and a MESSENGER

  CASSIUS. He is all the seas of him. That will not speak.
    I had reputed this is the state,
    Will now survey, the fatal sea of him.

    Enter two or three SERVINGMAN shall be trumpeter

  KING RICHARD. What will your Grace say so? Will you not stole
    With thankfulness as the wind

In [154]:
print(generate_text(model, 1000, seed='be ', k=3))

be the
    cause.
  CASSIUS. And the story of that time to speak of me.
  CASSIUS. I would this man, I will not send him to my heart,  
    That we will strike a way, and she was done
    To be the cause of this this conscience's house.
    I have a shame than we are most assay.
    I would have set thee to all my soul to me.
    To me thou shalt not bear thy honour for me.
    Then to thy season, what a pardon were
    A part of men are thought and be to stay,
    That shall be so they shall be strike of thee.
    I want not with thy heart that I was seen.
    The more thy fair did share, I hold thee so.
    I am thy sovereign shall be to be meet
    That thy brothers are better thoughts on me,
    When I am told to make thine eye and do.
    I would I had no more to thy bosom toward
    The secret of myself.
  KING HENRY. I am so sorry to the word, my lord.
  KING RICHARD. I would I shall be seen, my lord, and he was
    sent to the King.  
  KING HENRY. Why, there is nothing straigh